# Generate Synthetic Fourier Signal Dataset v1.0.0

This notebook creates 150,000 labeled one-dimensional signals from five piecewise-smooth families across clean, 30 dB, 20 dB, 10 dB, and 0 dB noise conditions. It runs the versioned CUDA generator, validates every generated shard, creates checksums, and packages the artifacts.

The dataset is written to the transient Colab path `/content/synthetic_fourier_noise_v1.0.0`. Download the release ZIP before disconnecting the runtime, or adapt the final cell to copy it to your own Drive.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/abbass12/FourierSeriesClassification.git'
REPO_DIR = Path('/content/FourierSeriesClassification')
OUTPUT_DIR = Path('/content/synthetic_fourier_noise_v1.0.0')
ARCHIVE_PATH = Path('/content/synthetic_fourier_noise_v1.0.0.zip')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Python:', sys.version.split()[0])

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a T4 GPU runtime in Runtime > Change runtime type.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
!python dataset_generation/generate_synthetic_fourier_dataset.py --output-dir {OUTPUT_DIR} --device cuda
!python dataset_generation/validate_synthetic_fourier_dataset.py --dataset-dir {OUTPUT_DIR}
!du -sh {OUTPUT_DIR}
!cat {OUTPUT_DIR}/validation_report.md

In [ ]:
if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()
shutil.make_archive(str(ARCHIVE_PATH.with_suffix('')), 'zip', OUTPUT_DIR.parent, OUTPUT_DIR.name)
!sha256sum {ARCHIVE_PATH}
print('Archive ready:', ARCHIVE_PATH, f'({ARCHIVE_PATH.stat().st_size / 2**30:.2f} GiB)')

In [ ]:
# Optional: download the complete compressed dataset archive to your computer.
from google.colab import files
files.download(str(ARCHIVE_PATH))